# 01 — Extraction des données brutes

Ce notebook télécharge le dataset **RLCS 2021-22** depuis Kaggle et le place dans `data/raw/`.

**Source :** [RLCS 2021-22 — Kaggle](https://www.kaggle.com/datasets/dylanmonfret/rlcs-202122)

**Pipeline ETL :**
```
01_extract (ce notebook)    Kaggle → data/raw/*.csv
02_transform                data/raw/*.csv → data/processed/ (nettoyage, normalisation)
03_load                     data/processed/ → DuckDB via SQLAlchemy + Alembic
```

## 1. Configuration

In [ ]:
import pandas as pd

from src.config import RAW_DIR
from src.etl.extract.config import KAGGLE_DATASET, PRIMARY_KEYS

print(f'Dossier cible    : {RAW_DIR}')
print(f'Dataset Kaggle   : {KAGGLE_DATASET}')

## 2. Téléchargement depuis Kaggle

Utilise `kagglehub` pour télécharger le dataset.
Si les CSV sont déjà présents dans `data/raw/`, l'étape est ignorée.

In [ ]:
from src.etl.extract.chore import ensure_raw_csvs

skipped, csv_paths = ensure_raw_csvs(RAW_DIR)

if skipped:
    print(f'CSV déjà présents ({len(csv_paths)} fichiers) — téléchargement ignoré.')
else:
    print(f'Fichiers copiés dans {RAW_DIR} :')
    for p in csv_paths:
        print(f'  → {p.name}')

## 3. Inventaire des fichiers

In [ ]:
from src.etl.extract.utils import inventory_all

csv_files = sorted(RAW_DIR.glob('*.csv'))
df_inv = inventory_all(RAW_DIR)
display(df_inv)
print(f'Total : {df_inv["lignes"].sum():,} lignes | {df_inv["taille_mb"].sum():.1f} MB')

## 4. Aperçu de chaque fichier

3 premières lignes + liste des colonnes par catégorie.

In [ ]:
for f in csv_files:
    row = df_inv[df_inv['fichier'] == f.name].iloc[0]
    df = pd.read_csv(f, nrows=3)
    print(f'\n{"=" * 60}')
    print(f'{f.name}  ({row["colonnes"]} colonnes, {row["lignes"]:,} lignes)')
    print(f'{"=" * 60}')
    print(f'Colonnes : {list(df.columns[:12])}{" ..." if len(df.columns) > 12 else ""}')
    display(df)

## 5. Qualité des données

Vérification rapide : valeurs nulles, types, doublons sur les clés.

In [ ]:
from src.etl.extract.utils import validate_all

results = validate_all(RAW_DIR, PRIMARY_KEYS)

for name, info in results.items():
    print(f'\n--- {name} ---')

    if info['pk_columns']:
        n = info['duplicates']
        print(f'  PK {info["pk_columns"]} : {n} doublons{"  !!" if n else ""}')

    nulls = info['nulls']
    if len(nulls) > 0:
        print(f'  Colonnes avec nulls ({len(nulls)}) :')
        for col, pct in nulls.head(5).items():
            print(f'    {col:<45} {pct:>5.1f}%')
        if len(nulls) > 5:
            print(f'    ... et {len(nulls) - 5} autres')
    else:
        print(f'  Aucun null')

## 6. Résumé

Les données brutes sont extraites dans `data/raw/`. Prochaine étape : **02_transform** (nettoyage et normalisation).

In [ ]:
print('Extraction termin\u00e9e.')
print(f'  Fichiers : {len(csv_files)} CSV dans {RAW_DIR}')
print(f'  Volume   : {df_inv["lignes"].sum():,} lignes | {df_inv["taille_mb"].sum():.1f} MB')
print(f'\nProchaine \u00e9tape : 02_transform.ipynb')